# Demo de RAG local con Ollama

Notebook de ejemplo para enseñar un flujo RAG básico usando solo Ollama en local. Cada alumno puede elegir el modelo generativo y el modelo de embeddings que mejor le funcione en su equipo.

Referencias oficiales:
- API base de Ollama: https://docs.ollama.com/api
- Generación: https://docs.ollama.com/api/generate
- Embeddings: https://docs.ollama.com/api/embed

## Celda 1. Instalación

In [27]:
#!pip install ollama numpy

## Celda 2. Comprobar que Ollama está disponible

Antes de seguir, asegúrate de tener Ollama arrancado en local y al menos dos modelos descargados:

- un modelo generativo, por ejemplo `gemma3`, `qwen3`, `llama3.1` o similar
- un modelo de embeddings, por ejemplo `embeddinggemma`, `qwen3-embedding` o `all-minilm`

In [1]:
import ollama


print(ollama.list())

models=[Model(model='gemma4:latest', modified_at=datetime.datetime(2026, 4, 10, 20, 48, 50, 767862, tzinfo=TzInfo(7200)), digest='85b0da114b956d28139d29e5639d57d497133793e00ce6352bec27a0869184a0', size=24869, details=ModelDetails(parent_model='', format='', family='', families=None, parameter_size='', quantization_level='')), Model(model='gemma4:31b-cloud', modified_at=datetime.datetime(2026, 4, 10, 20, 21, 45, 278952, tzinfo=TzInfo(7200)), digest='c382fbfbc73b6fdd08c8549c23caedc6e62eb09933c65a1fb82dbf3398320a4e', size=342, details=ModelDetails(parent_model='', format='', family='', families=None, parameter_size='', quantization_level='')), Model(model='nemotron-3-super:cloud', modified_at=datetime.datetime(2026, 3, 26, 12, 42, 27, 671391, tzinfo=TzInfo(3600)), digest='be3943c5a818be61a08f3563b971e392bfc12e506e296fb186c870f5c63377a4', size=345, details=ModelDetails(parent_model='', format='', family='', families=None, parameter_size='', quantization_level='')), Model(model='rnj-1:8b-cl

## Celda 3. Elegir modelos

Cambia estos nombres por los modelos que tengas instalados.

In [16]:
GEN_MODEL = "gemma4:31b-cloud" #gemma3
EMBED_MODEL =  "embeddinggemma"

In [9]:
import os
# get path of the current notebook
notebook_path = os.path.abspath("16-demo-rag-sencillo.ipynb")
print(notebook_path)

/home/jmsa/IESRafaelAlberti/RafaelAlberti25_26/Modulos/PIA/UD5/16-demo-rag-sencillo.ipynb


## Celda 4. Cargar documentos

In [11]:
from pathlib import Path

docs_path = Path("Documentacion/documentos_demo")
files = sorted(list(docs_path.glob("*.md")))
print([f.name for f in files])

['evaluacion.md', 'faq.md', 'proyecto.md']


## Celda 5. Leer textos

In [12]:
documents = []

for file in files:
    text = file.read_text(encoding="utf-8")
    documents.append({"source": file.name, "text": text})

for doc in documents:
    print(doc["source"], len(doc["text"]))

evaluacion.md 1018
faq.md 953
proyecto.md 1007


## Celda 6. Fragmentación básica

Aquí usamos una fragmentación muy simple por tamaño fijo. Es suficiente para la demo y además permite explicar después sus limitaciones.

In [14]:
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += max(1, chunk_size - overlap)
    return chunks

chunks = []
for doc in documents:
    for i, chunk in enumerate(chunk_text(doc["text"])):
        chunks.append({
            "source": doc["source"],
            "chunk_id": i,
            "text": chunk
        })

print(f"Total de fragmentos: {len(chunks)}")
print(chunks[0]["source"])
print(chunks[0]["text"][:300])

Total de fragmentos: 9
evaluacion.md
# Evaluación del proyecto

## Criterios generales

La evaluación del proyecto tendrá en cuenta tanto el resultado final como la justificación técnica de las decisiones tomadas.

## Qué se valorará

- Definición clara del problema.
- Coherencia entre objetivos, datos y solución propuesta.
- Calidad t


## Celda 7. Generar embeddings

In [17]:
texts = [chunk["text"] for chunk in chunks]

embed_response = ollama.embed(
    model=EMBED_MODEL,
    input=texts
)

embeddings = embed_response["embeddings"]
print(f"Embeddings generados: {len(embeddings)}")
print(f"Dimensión del vector: {len(embeddings[0])}")

Embeddings generados: 9
Dimensión del vector: 768


## Celda 8. Búsqueda por similitud

In [18]:
import numpy as np

def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def retrieve(query, top_k=3):
    query_embedding = ollama.embed(model=EMBED_MODEL, input=query)["embeddings"][0]
    scored = []
    for chunk, emb in zip(chunks, embeddings):
        score = cosine_similarity(query_embedding, emb)
        scored.append((score, chunk))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]

## Celda 9. Probar recuperación

In [19]:
query = "¿Qué se valora en la evaluación del proyecto?"
results = retrieve(query, top_k=3)

for score, chunk in results:
    print("=" * 80)
    print(f"Score: {score:.4f} | Fuente: {chunk['source']} | Chunk: {chunk['chunk_id']}")
    print(chunk["text"][:500])

Score: 0.6767 | Fuente: evaluacion.md | Chunk: 0
# Evaluación del proyecto

## Criterios generales

La evaluación del proyecto tendrá en cuenta tanto el resultado final como la justificación técnica de las decisiones tomadas.

## Qué se valorará

- Definición clara del problema.
- Coherencia entre objetivos, datos y solución propuesta.
- Calidad técnica de la implementación.
- Capacidad para justificar herramientas, modelos y arquitectura.
- Presentación ordenada de resultados y limitaciones.

## Fases de trabajo

1. Planteamiento del problema
Score: 0.4946 | Fuente: faq.md | Chunk: 1
una API externa si encaja mejor por tiempo, recursos o facilidad de integración.

## ¿Qué pasa si una categoría no aplica al proyecto?

Debe explicarse con claridad. También es una decisión técnica válida.

## ¿Se valora más la complejidad o la coherencia?

Se valora más la coherencia. Un stack pequeño y bien justificado suele ser mejor que una arquitectura ambiciosa mal integrada.

## ¿La evaluación exig

## Celda 10. Construir contexto para el modelo

In [20]:
def build_context(results):
    blocks = []
    for score, chunk in results:
        blocks.append(
            f"Fuente: {chunk['source']} | Fragmento: {chunk['chunk_id']}\n{chunk['text']}"
        )
    return "\n\n---\n\n".join(blocks)

context = build_context(results)
print(context[:1200])

Fuente: evaluacion.md | Fragmento: 0
# Evaluación del proyecto

## Criterios generales

La evaluación del proyecto tendrá en cuenta tanto el resultado final como la justificación técnica de las decisiones tomadas.

## Qué se valorará

- Definición clara del problema.
- Coherencia entre objetivos, datos y solución propuesta.
- Calidad técnica de la implementación.
- Capacidad para justificar herramientas, modelos y arquitectura.
- Presentación ordenada de resultados y limitaciones.

## Fases de trabajo

1. Planteamiento del problema

---

Fuente: faq.md | Fragmento: 1
una API externa si encaja mejor por tiempo, recursos o facilidad de integración.

## ¿Qué pasa si una categoría no aplica al proyecto?

Debe explicarse con claridad. También es una decisión técnica válida.

## ¿Se valora más la complejidad o la coherencia?

Se valora más la coherencia. Un stack pequeño y bien justificado suele ser mejor que una arquitectura ambiciosa mal integrada.

## ¿La evaluación exige despliegue compl

## Celda 11. Generar respuesta con contexto recuperado

In [21]:
prompt = f"""
Responde solo con información contenida en el contexto.
Si no encuentras la respuesta exacta, di que no aparece claramente en los documentos.

Contexto:
{context}

Pregunta: {query}
"""

response = ollama.generate(
    model=GEN_MODEL,
    prompt=prompt,
    stream=False
)

print(response["response"])

Se valorará lo siguiente:
- Definición clara del problema.
- Coherencia entre objetivos, datos y solución propuesta.
- Calidad técnica de la implementación.
- Capacidad para justificar herramientas, modelos y arquitectura.
- Presentación ordenada de resultados y limitaciones.

Además, se valorará más la coherencia que la complejidad.


## Celda 12. Comparar con respuesta sin RAG

In [22]:
no_rag_response = ollama.generate(
    model=GEN_MODEL,
    prompt=query,
    stream=False
)

print(no_rag_response["response"])

La evaluación de un proyecto es un proceso multidimensional. Dependiendo de la etapa en la que se encuentre (antes de iniciar, durante la ejecución o al finalizar), se valorarán aspectos distintos.

De manera general, los evaluadores buscan responder a tres preguntas fundamentales: **¿Es viable?**, **¿Se cumplieron los objetivos?** y **¿Valió la pena la inversión?**

Aquí te detallo los puntos clave que se valoran divididos por dimensiones:

---

### 1. Valoración Técnica y de Gestión (¿Se hizo bien?)
Se analiza la eficiencia operativa y el cumplimiento de la planificación.

*   **Cumplimiento de Plazos:** ¿Se entregó el proyecto en la fecha acordada? ¿Hubo desviaciones en el cronograma?
*   **Gestión de Recursos:** ¿Se utilizaron los materiales, el personal y el presupuesto de manera eficiente? ¿Hubo desperdicios o subutilización?
*   **Calidad de los Entregables:** ¿El producto final cumple con las especificaciones técnicas requeridas? ¿Funciona correctamente?
*   **Gestión de Riesgo

## Celda 13. Actividad para el alumnado

In [23]:
query = "¿Cuándo puede quedarse corto el chunking clásico?"
results = retrieve(query, top_k=3)
context = build_context(results)

prompt = f"""
Responde solo con información contenida en el contexto.
Si no encuentras la respuesta exacta, di que no aparece claramente en los documentos.

Contexto:
{context}

Pregunta: {query}
"""

response = ollama.generate(model=GEN_MODEL, prompt=prompt, stream=False)
print(response["response"])

No aparece claramente en los documentos.


## Celda 14. Ideas para ampliar la demo

- cambiar el modelo generativo,
- cambiar el modelo de embeddings,
- subir o bajar `top_k`,
- probar preguntas más difíciles,
- comparar este flujo con `12-recuperacion-avanzada-rag.md`.